In [1]:

import torch
from transformers import BertForSequenceClassification, BertTokenizer

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import Counter

from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    ConfusionMatrixDisplay
)

MODEL_PATH = r"D:\ecom-sentiment-stream\models\bert_finetuned"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bert_model = BertForSequenceClassification.from_pretrained(MODEL_PATH)
bert_tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)

bert_model.to(device)
bert_model.eval()

print("Saved BERT model loaded successfully!")
print("Device:", device)
print("Labels:", bert_model.config.id2label)

d:\ecom-sentiment-stream\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 524.68it/s]

Saved BERT model loaded successfully!
Device: cpu
Labels: {0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3', 4: 'LABEL_4', 5: 'LABEL_5', 6: 'LABEL_6', 7: 'LABEL_7', 8: 'LABEL_8', 9: 'LABEL_9', 10: 'LABEL_10', 11: 'LABEL_11', 12: 'LABEL_12', 13: 'LABEL_13', 14: 'LABEL_14', 15: 'LABEL_15', 16: 'LABEL_16', 17: 'LABEL_17', 18: 'LABEL_18', 19: 'LABEL_19', 20: 'LABEL_20', 21: 'LABEL_21', 22: 'LABEL_22', 23: 'LABEL_23', 24: 'LABEL_24', 25: 'LABEL_25', 26: 'LABEL_26', 27: 'LABEL_27', 28: 'LABEL_28', 29: 'LABEL_29', 30: 'LABEL_30', 31: 'LABEL_31', 32: 'LABEL_32', 33: 'LABEL_33', 34: 'LABEL_34', 35: 'LABEL_35', 36: 'LABEL_36', 37: 'LABEL_37', 38: 'LABEL_38', 39: 'LABEL_39', 40: 'LABEL_40', 41: 'LABEL_41', 42: 'LABEL_42', 43: 'LABEL_43', 44: 'LABEL_44', 45: 'LABEL_45', 46: 'LABEL_46', 47: 'LABEL_47', 48: 'LABEL_48', 49: 'LABEL_49', 50: 'LABEL_50', 51: 'LABEL_51', 52: 'LABEL_52', 53: 'LABEL_53', 54: 'LABEL_54', 55: 'LABEL_55', 56: 'LABEL_56', 57: 'LABEL_57', 58: 'LABEL_58', 59: 'LABEL_59', 6

In [2]:
import pandas as pd

DATA_PATH = r"D:\ecom-sentiment-stream\data\Labeled_Amazon_reviews.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)


(21214, 11)


In [3]:
X=df["Review Text"]
Y=df["label"]

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.30, random_state=42)


In [8]:
from transformers import BertTokenizer

MODEL_PATH = r"D:\ecom-sentiment-stream\models\bert_finetuned"

tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [9]:
class NewsDataset(Dataset):

    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __getitem__(self, idx):

        text = self.texts[idx] if isinstance(self.texts[idx], str) else ""

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=256,
            return_tensors="pt"
        )

        item = {
            k: v.squeeze(0)
            for k, v in encoding.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return item

    def __len__(self):
        return len(self.labels)

In [10]:
train_dataset = NewsDataset(
    list(X_train),
    list(y_train)
)

test_dataset = NewsDataset(
    list(X_test),
    list(y_test)
)

In [14]:
from torch.utils.data import DataLoader

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

print("Test DataLoader created!")
print("Test samples:", len(test_dataset))
print("Number of batches:", len(test_loader))

Test DataLoader created!
Test samples: 6365
Number of batches: 398


In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

bert_model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for batch in test_loader:  # Corrected here

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))

print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_preds,
    labels=[0, 1, 2],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds, labels=[0, 1, 2]))

KeyboardInterrupt: 

In [ ]:
accuracy = accuracy_score(all_labels, all_preds)

precision = precision_score(
    all_labels, all_preds,
    average="macro",
    zero_division=0
)

recall = recall_score(
    all_labels, all_preds,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    all_labels, all_preds,
    average="macro",
    zero_division=0
)

print("Accuracy:", round(accuracy, 4))
print("Macro Precision:", round(precision, 4))
print("Macro Recall:", round(recall, 4))
print("Macro F1:", round(f1, 4))

In [ ]:
print("\nClassification Report:")

print(classification_report(
    all_labels,
    all_preds,
    labels=[0, 1, 2],
    target_names=["Negative", "Neutral", "Positive"],
    zero_division=0
))

print("\nConfusion Matrix:")

cm = confusion_matrix(
    all_labels,
    all_preds,
    labels=[0, 1, 2]
)

print(cm)